# Lab 4.1: Training Transformers for Remote Sensing Classification

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This notebook demonstrates training a Transformer model for remote sensing land cover classification using PyTorch Lightning with distributed training support.

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 3.1 | Data Preprocessing | ✅ Previous |
| Lab 3.2 | Google Earth Engine Acquisition | ✅ Previous |
| Lab 4 | Understanding Transformers | ✅ Previous |
| Lab 4.1 | **Training on Sentinel-2 Data** | 🔄 **Current** |
| Lab 5 | Distributed Training (Multi-GPU) | ⬜ Next |
| Lab 6 | Validation & Performance Metrics | ⬜ Next |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Final |

---

## Project Context

**Inputs (From Lab 3.1):**
- Sentinel-2 patches (10 spectral bands)
- CORINE labels (12 land cover classes)
- CSV format: `trainSet1_cleaned.csv` and `valSet1_cleaned.csv`

**What You'll Do:**
- Build and train a transformer-based classifier
- Handle multi-GPU training with PyTorch Lightning
- Deploy on HPC with Slurm batch scripts
- Generate model checkpoints for evaluation

**Output:**
- Trained model weights
- Training logs and validation metrics
- Ready for deployment in Lab 6 (evaluation)

---

## Overview

In this lab, you will:
1. **Build a Transformer Model**: Create a transformer architecture for classification
2. **Prepare Data**: Load and preprocess remote sensing data
3. **Train with PyTorch Lightning**: Use Lightning for simplified training loops
4. **Distributed Training**: Scale training across multiple GPUs and nodes
5. **HPC Deployment**: Submit jobs to Slurm-managed HPC systems

## Part 1: Setup and Dependencies

First, import the required libraries.

In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np

Matplotlib created a temporary cache directory at /tmp/matplotlib-acpc4jdb because the default path (/p/home/jusers/hashim1/jureca/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Part 2: Define the Transformer Model

We'll use PyTorch Lightning to create a transformer-based classifier. The model uses multi-head self-attention to capture spatial relationships in remote sensing data.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

class TransformerModel(pl.LightningModule):
    """
    Transformer encoder classifier for land cover classification.

    Input per sample:
      - either flattened: (B, 36)
      - or patch:         (B, 3, 3, 4)

    We interpret a 3x3 patch as 9 tokens (pixels),
    each token has 4 features (Sentinel-2 bands: B02,B03,B04,B08).
    """
    def __init__(
        self,
        num_classes: int = 26,
        bands_dim: int = 4,
        seq_len: int = 9,
        d_model: int = 64,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 256,
        lr: float = 1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.num_classes = num_classes
        self.bands_dim = bands_dim
        self.seq_len = seq_len
        self.d_model = d_model
        self.lr = lr

        # Project 4-band pixel vectors -> d_model
        self.input_proj = nn.Linear(bands_dim, d_model)

        # Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Classifier head
        self.classifier = nn.Linear(d_model, num_classes)

    def _to_tokens(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert x to (B, seq_len, bands_dim).
        Accepts:
          - (B, 36) -> (B, 9, 4)
          - (B, 3, 3, 4) -> (B, 9, 4)
          - already (B, 9, 4) -> unchanged
        """
        if x.dim() == 2:  # (B, 36)
            x = x.view(x.size(0), self.seq_len, self.bands_dim)
        elif x.dim() == 4:  # (B, 3, 3, 4)
            x = x.view(x.size(0), self.seq_len, self.bands_dim)
        elif x.dim() == 3:  # (B, 9, 4)
            pass
        else:
            raise ValueError(f"Unexpected x shape: {tuple(x.shape)}")
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns logits of shape (B, num_classes)
        """
        x = x.to(torch.float32)
        x = self._to_tokens(x)              # (B, 9, 4)
        x = self.input_proj(x)              # (B, 9, d_model)
        x = self.encoder(x)                 # (B, 9, d_model)

        # Pool over tokens (mean pooling)
        x = x.mean(dim=1)                   # (B, d_model)

        logits = self.classifier(x)         # (B, num_classes)
        return logits

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log("val_loss", loss, prog_bar=True)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", acc, prog_bar=True)
        return loss



    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

## Part 3: Custom Dataset Class

Create a PyTorch Dataset for loading remote sensing data.

In [3]:
class YourCustomDataset(Dataset):
    """
    Custom Dataset for remote sensing data.
    
    Parameters:
    -----------
    data : numpy.ndarray
        Feature data (n_samples, n_features)
    labels : numpy.ndarray
        Target labels (n_samples,)
    """
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.labels[idx]
        return x, y

## Part 4: Data Loading and Preprocessing

Load the training and validation datasets from CSV files.

In [22]:
import numpy as np

# --- 1) Load from your .npz ---
npz_path = "/p/scratch/training2600/hashim1/training_data/combined_training_data.npz"
data = np.load(npz_path)

print("Keys in npz:", list(data.keys()))

if "patches" in data and "labels" in data:
    X = data["patches"]
    y = data["labels"]
else:
    possible_X_keys = ["X", "acquisitions", "inputs", "features", "patch"]
    possible_y_keys = ["y", "targets", "labels", "corine", "target"]

    X_key = next((k for k in possible_X_keys if k in data), None)
    y_key = next((k for k in possible_y_keys if k in data), None)

    if X_key is None or y_key is None:
        raise KeyError(
            f"Could not find (patches, labels) in {npz_path}. "
            f"Available keys: {list(data.keys())}"
        )

    X = data[X_key]
    y = data[y_key]

print(f"Loaded X shape: {X.shape}, dtype={X.dtype}")
print(f"Loaded y shape: {y.shape}, dtype={y.dtype}")

# --- 2) Preprocess like your CSV pipeline did ---
X = X.astype(np.float32) * 0.0001
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")

# Optional but recommended for CrossEntropy:
# If your labels are 1..K, shift to 0..K-1.
# Uncomment if needed:
# if y.min() == 1:
#     y = y.astype(np.int64) - 1

# --- 3) Split into train / val / test ---
def make_splits_train_val_test(
    X,
    y,
    train_frac=0.7,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
    stratify=True,
):
    """
    Train/Val/Test split with optional stratification.

    Requirements:
      train_frac + val_frac + test_frac == 1.0
    """
    total = train_frac + val_frac + test_frac
    assert abs(total - 1.0) < 1e-6, "train_frac + val_frac + test_frac must be 1.0"

    rng = np.random.default_rng(seed)
    n = len(y)

    if not stratify:
        idx = rng.permutation(n)
        n_train = int(n * train_frac)
        n_val = int(n * val_frac)

        train_idx = idx[:n_train]
        val_idx = idx[n_train : n_train + n_val]
        test_idx = idx[n_train + n_val :]
        return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])

    # Stratified: do per-class splitting
    classes, y_inv = np.unique(y, return_inverse=True)

    train_idx_all, val_idx_all, test_idx_all = [], [], []

    for c in range(len(classes)):
        cls_idx = np.where(y_inv == c)[0]
        cls_idx = rng.permutation(cls_idx)

        n_c = len(cls_idx)
        n_train_c = int(n_c * train_frac)
        n_val_c = int(n_c * val_frac)

        train_idx_all.append(cls_idx[:n_train_c])
        val_idx_all.append(cls_idx[n_train_c : n_train_c + n_val_c])
        test_idx_all.append(cls_idx[n_train_c + n_val_c :])

    train_idx = rng.permutation(np.concatenate(train_idx_all))
    val_idx = rng.permutation(np.concatenate(val_idx_all))
    test_idx = rng.permutation(np.concatenate(test_idx_all))

    return (X[train_idx], y[train_idx]), (X[val_idx], y[val_idx]), (X[test_idx], y[test_idx])


(X_train, y_train), (X_val, y_val), (X_test, y_test) = make_splits_train_val_test(
    X_flat,
    y,
    train_frac=0.8,
    val_frac=0.1,
    test_frac=0.1,
    seed=42,
    stratify=False,
)

print(f"Training samples:   {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples:       {X_test.shape[0]}")
print(f"Features/sample:    {X_train.shape[1]}")

# --- 4) Optional: sanity check label distribution ---
def label_hist(y_arr, name, topk=10):
    vals, cnts = np.unique(y_arr, return_counts=True)
    order = np.argsort(cnts)[::-1]
    print(f"\n{name} label distribution (top {topk}):")
    for v, c in zip(vals[order][:topk], cnts[order][:topk]):
        print(f"  {int(v):3d}: {int(c)}")

label_hist(y_train, "Train")
label_hist(y_val, "Val")
label_hist(y_test, "Test")

Keys in npz: ['patches', 'labels']
Loaded X shape: (200000, 3, 3, 4), dtype=float32
Loaded y shape: (200000,), dtype=uint8
Flattened X shape: (200000, 36)
Training samples:   160000
Validation samples: 20000
Test samples:       20000
Features/sample:    36

Train label distribution (top 10):
   12: 74377
   24: 29969
   18: 29500
    2: 10051
   21: 7439
   25: 4249
   20: 1955
    3: 1530
   11: 742
    1: 188

Val label distribution (top 10):
   12: 9310
   24: 3816
   18: 3651
    2: 1255
   21: 911
   25: 515
   20: 239
    3: 202
   11: 76
    1: 25

Test label distribution (top 10):
   12: 9384
   18: 3714
   24: 3658
    2: 1243
   21: 943
   25: 480
   20: 258
    3: 212
   11: 85
    1: 23


## Part 5: Initialize Model and Data Loaders

Create the model and data loaders for training.

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

class MLP(pl.LightningModule):
    def __init__(self, in_dim=36, num_classes=26, lr=1e-3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )
        self.lr = lr

    def forward(self, x):
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
        return self.net(x.float())

    def training_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        y = y.long()
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

In [24]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler

y_train_np = np.asarray(y_train, dtype=np.int64)

classes, counts = np.unique(y_train_np, return_counts=True)

print("Train classes:", classes)
print("Counts:", counts)

# Inverse-frequency weights per class value (not per index)
class_weight_by_value = {int(c): float(1.0 / n) for c, n in zip(classes, counts)}

# Map each sample label to its weight
sample_weights = np.array([class_weight_by_value[int(lbl)] for lbl in y_train_np], dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

print("Sampler created. Example weights:", sample_weights[:10])

Train classes: [ 1  2  3 11 12 18 20 21 24 25]
Counts: [  188 10051  1530   742 74377 29500  1955  7439 29969  4249]
Sampler created. Example weights: [1.34450166e-05 3.33678134e-05 3.33678134e-05 1.34450166e-05
 3.38983051e-05 2.35349494e-04 3.33678134e-05 1.34450166e-05
 1.34450166e-05 3.33678134e-05]


In [25]:
# Hyperparameters
batch_size = 512

# Initialize model
model = TransformerModel(lr=1e-4)

# Create datasets
train_dataset = YourCustomDataset(X_train, y_train)
val_dataset = YourCustomDataset(X_val, y_val)
test_dataset = YourCustomDataset(X_test, y_test)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=512,
    sampler=sampler,
    shuffle=False,        # must be False when sampler is set
    num_workers=2,
    pin_memory=True,
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

Training batches: 313
Validation batches: 40


/tmp/ipykernel_50085/1062129976.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


## Part 6: Training with PyTorch Lightning

### Single GPU Training

For local development or single GPU training:

In [26]:
xb, yb = next(iter(train_dataloader))
print("x:", xb.shape, xb.dtype)
print("y:", yb.shape, yb.dtype)
print("y min/max:", yb.min().item(), yb.max().item())
print("unique y sample:", torch.unique(yb)[:20])

# also check num_classes
print("model.num_classes:", model.num_classes)

x: torch.Size([512, 36]) torch.float32
y: torch.Size([512]) torch.uint8
y min/max: 1 25
unique y sample: tensor([ 1,  2,  3, 11, 12, 18, 20, 21, 24, 25], dtype=torch.uint8)
model.num_classes: 26


In [27]:
# Train on single GPU (for testing/development)
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=60,  # Use fewer epochs for testing
    gradient_clip_val=1.0,
    log_every_n_steps=2,
)

# Start training
trainer.fit(model, train_dataloader, val_dataloader)
trainer.test(model, dataloaders=test_dataloader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ input_proj │ Linear             │    320 │ train │     0 │
│ 1 │ encoder    │ TransformerEncoder │  199 K │ train │     0 │
│ 2 │ classifier │ Linear             │  1.7 K │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 201 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 201 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 44                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

SLURM auto-requeueing enabled. Setting signal handlers.


Output()

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3     ]
SLURM auto-requeueing enabled. Setting signal handlers.


Output()

/p/project1/training2600/hashim1/envs/ml_eo_course/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │   0.012900000438094139    │
│         test_loss         │    2.3184170722961426     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 2.3184170722961426, 'test_acc': 0.012900000438094139}]

In [14]:
def majority_baseline(y, name):
    vals, cnts = np.unique(y, return_counts=True)
    p = cnts.max() / cnts.sum()
    v = vals[np.argmax(cnts)]
    print(f"{name}: n={len(y)} classes={len(vals)}")
    print(f"  majority label={v}, proportion={p:.3f}")
    print(f"  top-5:", sorted(zip(vals, cnts), key=lambda x: -x[1])[:5])

majority_baseline(y_train, "train")
majority_baseline(y_val, "val")
majority_baseline(y_test, "test")

train: n=159996 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 74456), (24, 29954), (18, 29492), (2, 10039), (21, 7434)]
val: n=19996 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 9307), (24, 3744), (18, 3686), (2, 1254), (21, 929)]
test: n=20008 classes=10
  majority label=12, proportion=0.465
  top-5: [(12, 9308), (24, 3745), (18, 3687), (2, 1256), (21, 930)]


## Part 7: Complete Training Script

Here's the complete training script that can be run as a standalone Python file for HPC submission:

In [ ]:
# This cell shows the complete script structure
# Save as train_transformer.py for HPC submission

"""
if __name__ == '__main__':
    # Load data from CSV files
    training_data = np.loadtxt("/p/project/training2328/lab4_1/data/trainSet1_cleaned.csv", 
                              delimiter=",", dtype=int)
    validation_data = np.loadtxt("/p/project/training2328/lab4_1/data/valSet1_cleaned.csv", 
                                delimiter=",", dtype=int)
    
    # Extract features and labels
    X_train = training_data[:, 1:] * 0.0001
    y_train = training_data[:, 0]
    X_val = validation_data[:, 1:] * 0.0001
    y_val = validation_data[:, 0]
    
    batch_size = 512
    num_gpus = int(os.environ['SLURM_NTASKS_PER_NODE'])
    num_nodes = int(os.environ['SLURM_JOB_NUM_NODES'])

    # Initialize model and datasets
    model = TransformerModel()
    train_dataset = YourCustomDataset(X_train, y_train)
    val_dataset = YourCustomDataset(X_val, y_val)

    # Initialize data loaders
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

    # Set up trainer with multi-GPU and multi-node support
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=num_gpus, 
        strategy="ddp", 
        num_nodes=num_nodes, 
        max_epochs=150
    )

    # Train the model
    trainer.fit(model, train_dataloader, val_dataloader)
"""
pass

## Part 8: HPC Deployment with Slurm

### Slurm Submission Script

To run the training on an HPC cluster, use the following Slurm batch script:

In [ ]:
%%bash
# Save this as submit_training.sh
# Submit with: sbatch submit_training.sh

#!/bin/bash
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-node=4
#SBATCH --exclusive
#SBATCH --account=training2328
#SBATCH --output=output.out
#SBATCH --error=error.er
#SBATCH --time=00:20:00
#SBATCH --job-name=JOAOsTorch
#SBATCH --gres=gpu:4 --partition=dc-gpu-devel

module --force purge
module use $OTHERSTAGES 
ml Stages/2023  GCC/11.3.0  OpenMPI/4.1.4
# load virtual environment if needed
source <your_venv>
export CUDA_VISIBLE_DEVICES=0,1,2,3

##### Number of total processes
echo " "
echo " Nodelist       := " $SLURM_JOB_NODELIST
echo " Number of nodes:= " $SLURM_JOB_NUM_NODES
echo " Ntasks per node:= " $SLURM_NTASKS_PER_NODE
echo " Ntasks         := " $SLURM_NTASKS
echo " "

echo ""
echo "Run started at:- "
date
srun --cpu-bind=none python -u train_transformer.py
echo "Run finished at:- "
date

### Submitting the Job

To submit your training job to the HPC cluster:

```bash
# Make sure the script is executable
chmod +x submit_training.sh

# Submit the job
sbatch submit_training.sh

# Monitor job status
squeue -u $USER

# View output
tail -f output.out

# Check for errors
tail -f error.er
```

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

## Summary

This notebook demonstrated:

1. **Transformer Architecture**: Built a multi-layer transformer with self-attention for remote sensing classification
2. **PyTorch Lightning**: Simplified training code with automatic optimization and logging
3. **Data Pipeline**: Custom Dataset and DataLoader for efficient data loading
4. **Distributed Training**: DDP strategy for multi-GPU and multi-node training
5. **HPC Integration**: Slurm batch scripts for submitting jobs to HPC clusters

The trained model can classify remote sensing data into 12 land cover categories using transformer-based deep learning.

---

## Next Steps

### Monitoring Your Training

After submitting your job with `sbatch submit_training.sh`:

```bash
# Check job status
squeue -u $USER

# Monitor output in real-time
tail -f output.out

# Once training completes, check outputs
ls -la checkpoints/
```

### Checkpoint Management

PyTorch Lightning saves checkpoints during training:
- Best model based on validation loss: `checkpoints/best_model.ckpt`
- Last model: `checkpoints/last.ckpt`
- Use for evaluation in Lab 6

### Preparing for Lab 5: Distributed Training

If you want to scale to more GPUs/nodes:

1. **Modify Slurm script**:
   ```bash
   #SBATCH --nodes=4          # Increase nodes
   #SBATCH --ntasks-per-node=8 # More GPUs per node
   #SBATCH --gpus-per-node=8
   ```

2. **PyTorch Lightning auto-handles DDP** - no code changes needed!

### Preparing for Lab 6: Validation & Evaluation

Save your best model for the next lab:

```bash
# Copy checkpoint to accessible location
cp checkpoints/best_model.ckpt ~/models/lab4_transformer.ckpt
```

In Lab 6, you'll:
- Load this checkpoint
- Run inference on validation set
- Calculate accuracy metrics (OA, PA, UA)
- Generate confusion matrices
- Compare with other land cover products (WorldCover, Esri)

---

## Troubleshooting

**Q: Training is slow on single GPU**
- Normal! Transformers are computationally intensive
- Move to Lab 5 for distributed training
- Reduce batch size if out of memory

**Q: Job gets killed with "OOM"**
- Reduce `batch_size` in the script
- Reduce number of transformer layers
- Increase time allocation: `#SBATCH --time=01:00:00`

**Q: How long should training take?**
- Single GPU: ~4-6 hours for 150 epochs
- 4 GPUs: ~1-2 hours
- 8 GPUs (Lab 5): ~30-45 minutes

---

## Key Configuration Parameters

### Model Architecture
- **Input dimension**: 10 features (spectral bands)
- **Attention heads**: 10 heads for multi-head attention
- **Transformer layers**: 5 encoder-decoder layers
- **Hidden dimension**: 254 for feedforward network
- **Output classes**: 12 land cover types

### Training Configuration
- **Batch size**: 512 samples per batch
- **Learning rate**: 0.01 (Adam optimizer)
- **Epochs**: 150 for full training
- **Loss function**: CrossEntropyLoss

### HPC Configuration
- **Nodes**: 2 compute nodes
- **GPUs per node**: 4 GPUs
- **CPUs per task**: 32 cores
- **Strategy**: Distributed Data Parallel (DDP)
- **Time limit**: 20 minutes (adjust for full training)

---

**Continue to Lab 5 for distributed training →**